# Restaurant Big Data Analytics Project
**End-to-end Big Data pipeline built on the Databricks Lakehouse Platform**

## Project Overview
This project ingests, cleans, transforms, and analyzes **11.11 million restaurant order transactions** spanning 6 years (2020–2025) across 6 branches in Egypt. The pipeline implements the **Medallion Architecture** (Bronze → Silver → Gold) using pure SQL on Delta Lake, and delivers business-ready KPI views consumable by Power BI.

## Architecture
| Stage | Layer | Purpose |
|-------|-------|---------|
| 1️⃣ | **Volume (`raw_files`)** | Landing zone for 9 source files (7 CSV + 2 JSON) |
| 2️⃣ | **Bronze** | Raw ingestion via `COPY INTO` with metadata columns |
| 3️⃣ | **Data Quality Report** | Profile data before transformation |
| 4️⃣ | **Silver** | Cleaned, unified, enriched orders |
| 4️⃣b | **Quarantine** | Invalid rows preserved for audit |
| 5️⃣ | **Gold** | Star schema: fact + dimension tables |
| 6️⃣ | **KPI Views** | Pre-aggregated business metrics |
| 7️⃣ | **Power BI** | Final dashboard layer (Phase 2) |

**Flow:** `Source Files → Volume → Bronze → DQ Report → Silver (+ Quarantine) → Gold → KPI Views → Power BI`

## Tech Stack
- **Databricks** — Lakehouse Platform (Free Edition, Serverless SQL)
- **Delta Lake** — ACID transactions, time travel, OPTIMIZE / ZORDER
- **Unity Catalog** — Three-level namespace (catalog.schema.table)
- **SQL** — All ETL implemented in pure SQL
- **Power BI** — Final visualization layer (Phase 2)

## Key Findings (Preview)

- **11.11M transactions** validated and processed
- **29.1% of source `total_amount` values were inaccurate** — corrected in the Silver layer
- **6 distinct branches** across Egypt (Cairo, Alexandria, Giza, Mansoura, Tanta, Asyut)
- **6-year date range** (2020-01-01 to 2025-12-30)
- Zero null `customer_id` values, zero invalid ratings

## Project Structure (Notebook Sections)

1. **Step 1** — Project Setup (Catalogs, Schemas, Volume, File Upload)
2. **Step 2** — Bronze Layer: Raw Ingestion
3. **Step 3** — Data Quality Report
4. **Step 4** — Silver Layer: Cleaning + Quarantine
5. **Step 5** — Silver Layer: Time Enrichment
6. **Step 6** — Gold Layer: Star Schema
7. **Step 7** — Gold Layer: Business KPI Views
8. **Step 8** — Performance Optimization (OPTIMIZE + ZORDER)
9. **Step 9** — Self-Documenting Tables (COMMENTS)
10. **Step 10** — Project Summary & Findings

In [0]:
-- STEP 1.1: Create the Medallion Architecture Schemas
-- Purpose: Set up Bronze, Silver, and Gold schemas inside the workspace catalog.
-- This implements the Medallion Architecture pattern, where each layer progressively refines the data:
--      Bronze = raw ingestion (faithful copy of source)
--      Silver = cleaned, unified, and validated
--      Gold = business-ready aggregates and KPIs
-- 
-- Why three separate schemas (not just three table prefixes)?
--      Clear separation of concerns
--      Different access permissions can be granted per layer
--      Catalog Explorer displays the architecture visually
--      Industry-standard pattern for the Lakehouse

-- Create the BRONZE schema (raw landing zone)
CREATE SCHEMA IF NOT EXISTS workspace.bronze
COMMENT 'Bronze layer: Raw data ingested from source files. No transformations applied.';

-- Create the SILVER schema (cleaned and unified)
CREATE SCHEMA IF NOT EXISTS workspace.silver
COMMENT 'Silver layer: Cleaned, deduplicated, type-cast, and unified data. Ready for modeling.';

-- Create the GOLD schema (business-ready)
CREATE SCHEMA IF NOT EXISTS workspace.gold
COMMENT 'Gold layer: Star schema (fact + dimension tables) and pre-aggregated KPI views for BI consumption.';

-- Confirm all three schemas were created successfully
SHOW SCHEMAS IN workspace;

databaseName
bronze
default
gold
information_schema
silver


In [0]:
-- STEP 1.2: Create the Volume for Raw Source Files
-- Purpose: Create a managed Volume inside the bronze schema. The Volume is the
--          landing zone where the 9 source files (7 CSV + 2 JSON) will be 
--          uploaded before ingestion into Bronze tables.
--
-- Why use a Volume?
--      Unity Catalog-native file storage (governance + access control)
--      Decouples file storage from table creation (we control ingestion timing)
--      Enables "COPY INTO" for idempotent, re-runnable ingestion
--      Files can be re-processed without re-uploading
--      Cleaner than legacy DBFS paths

-- Create a managed Volume named 'raw_files' inside the bronze schema
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_files
COMMENT 'Landing zone for raw source files (7 CSV + 2 JSON) before Bronze ingestion.';

-- Confirm the Volume was created
SHOW VOLUMES IN workspace.bronze;

database,volume_name
bronze,raw_files


In [0]:
LIST '/Volumes/workspace/bronze/raw_files/csv/';

path,name,size,modification_time
/Volumes/workspace/bronze/raw_files/csv/restaurant1.csv,restaurant1.csv,109888011,1778065057000
/Volumes/workspace/bronze/raw_files/csv/restaurant2.csv,restaurant2.csv,165370349,1778065057000
/Volumes/workspace/bronze/raw_files/csv/restaurant3.csv,restaurant3.csv,97779151,1778065054000
/Volumes/workspace/bronze/raw_files/csv/restaurant4.csv,restaurant4.csv,85682237,1778065046000
/Volumes/workspace/bronze/raw_files/csv/restaurant5.csv,restaurant5.csv,70298509,1778065033000
/Volumes/workspace/bronze/raw_files/csv/restaurant6.csv,restaurant6.csv,220889751,1778065057000
/Volumes/workspace/bronze/raw_files/csv/restaurant7.csv,restaurant7.csv,276381037,1778065057000


In [0]:
LIST '/Volumes/workspace/bronze/raw_files/json/';

path,name,size,modification_time
/Volumes/workspace/bronze/raw_files/json/restaurant_json1.json,restaurant_json1.json,303991187,1778065357000
/Volumes/workspace/bronze/raw_files/json/restaurant_json2.json,restaurant_json2.json,284372018,1778065357000


# Step 2: Bronze Layer: Raw Ingestion
The Bronze layer is the **landing zone** for raw data. We ingest all 9 source files (7 CSV + 2 JSON) from the Volume into Delta tables using the `COPY INTO` command.

### Why `COPY INTO`?
- **Idempotent** — if the cell is re-run, files already loaded are skipped automatically
- **Production-grade** — this is the modern Databricks pattern for file-based ingestion
- **Fully scripted** — no UI clicks, fully reproducible from this notebook alone

### Bronze layer principles
- **Faithful copy** — no transformations, no cleaning, no type changes (data preserved as-is from source)
- **Metadata enrichment** — every row gets two extra columns:
  - `_ingested_at` — timestamp of when the row was loaded into Bronze
  - `_source_file` — the source filename (data lineage)
- **One Bronze table per source format** — `bronze_orders_csv` for the 7 CSVs, `bronze_orders_json` for the 2 JSONs

We will UNION these two tables in the Silver layer (Step 4), not here.

In [0]:
-- STEP 2.1: Create Bronze tables with explicit schemas
-- Purpose: Define empty Bronze Delta tables with explicit column types BEFORE ingesting data. 
-- This is "schema-on-write" the production pattern recommended for the Lakehouse.
--
-- Why explicit schemas (not auto-inferred)?
--      Catches type mismatches at ingestion time (not silently in queries later)
--      Documents the source data contract directly in code
--      Prevents type drift across re-runs (auto-inference can change types)
--      Industry-standard for production data pipelines
--
-- Schema reference (validated via Python inspection):
--      15 columns identical across all 9 source files (7 CSV + 2 JSON)

-- Bronze table for CSV sources (7 files):
CREATE OR REPLACE TABLE workspace.bronze.bronze_orders_csv (
    order_id        BIGINT,
    order_date      STRING,           -- kept as STRING in Bronze; cast to DATE in Silver
    hour            INT,
    category        STRING,
    item_name       STRING,
    price           DOUBLE,
    quantity        INT,
    discount        DOUBLE,
    total_amount    DOUBLE,           -- raw value from source (may be inaccurate; corrected in Silver)
    branch          STRING,
    payment_method  STRING,
    order_type      STRING,
    customer_id     BIGINT,
    rating          INT,
    is_weekend      INT,
    -- Metadata columns added during ingestion (not in source files)
    _ingested_at    TIMESTAMP,        -- when this row was loaded into Bronze
    _source_file    STRING            -- source filename for data lineage
)
USING DELTA
COMMENT 'Bronze layer raw ingestion of CSV order files. Faithful copy of source data with ingestion metadata. No transformations applied.';

-- Bronze table for JSON sources (2 files) — same schema as CSV:
CREATE OR REPLACE TABLE workspace.bronze.bronze_orders_json (
    order_id        BIGINT,
    order_date      STRING,
    hour            INT,
    category        STRING,
    item_name       STRING,
    price           DOUBLE,
    quantity        INT,
    discount        DOUBLE,
    total_amount    DOUBLE,
    branch          STRING,
    payment_method  STRING,
    order_type      STRING,
    customer_id     BIGINT,
    rating          INT,
    is_weekend      INT,
    _ingested_at    TIMESTAMP,
    _source_file    STRING
)
USING DELTA
COMMENT 'Bronze layer raw ingestion of JSON order files. Faithful copy of source data with ingestion metadata. No transformations applied.';

-- Confirm both tables were created (should show 2 tables)
SHOW TABLES IN workspace.bronze;

database,tableName,isTemporary
bronze,bronze_orders_csv,false
bronze,bronze_orders_json,false


In [0]:
-- STEP 2.2: Ingest 7 CSV files into bronze_orders_csv
-- Purpose: Load all CSV files from the Volume into the Bronze CSV table using the COPY INTO command. 
-- Adds ingestion metadata (timestamp + source file) to enable data lineage tracking.
--
-- Key features of COPY INTO:
--      IDEMPOTENT: re-running this cell skips files already loaded (Databricks tracks ingested files internally).
--      New files added to the Volume later will be picked up automatically on the next run.
--      PARALLEL: all 7 files read concurrently by Spark
--      EXPLICIT TYPING: each column is CAST in the SELECT to match the target table schema exactly (avoids type-inference mismatches with the reader)
--
-- FORMAT_OPTIONS:
--      header = 'true'     -> first row of each CSV is column names

COPY INTO workspace.bronze.bronze_orders_csv
FROM (
    -- Cast each column explicitly to match the target table schema.
    -- This prevents Delta merge errors caused by reader type-inference mismatches.
    SELECT
        CAST(order_id       AS BIGINT)  AS order_id,
        CAST(order_date     AS STRING)  AS order_date,
        CAST(hour           AS INT)     AS hour,
        CAST(category       AS STRING)  AS category,
        CAST(item_name      AS STRING)  AS item_name,
        CAST(price          AS DOUBLE)  AS price,
        CAST(quantity       AS INT)     AS quantity,
        CAST(discount       AS DOUBLE)  AS discount,
        CAST(total_amount   AS DOUBLE)  AS total_amount,
        CAST(branch         AS STRING)  AS branch,
        CAST(payment_method AS STRING)  AS payment_method,
        CAST(order_type     AS STRING)  AS order_type,
        CAST(customer_id    AS BIGINT)  AS customer_id,
        CAST(rating         AS INT)     AS rating,
        CAST(is_weekend     AS INT)     AS is_weekend,
        CURRENT_TIMESTAMP()             AS _ingested_at,
        _metadata.file_name             AS _source_file
    FROM '/Volumes/workspace/bronze/raw_files/csv/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true'
);

-- Validate ingestion: count rows loaded into Bronze CSV table
-- Expected: 9,310,000 rows (sum of all 7 CSVs from inspection)
SELECT 
    COUNT(*) AS total_rows_loaded,
    COUNT(DISTINCT _source_file) AS distinct_source_files,
    MIN(_ingested_at) AS first_ingested_at,
    MAX(_ingested_at) AS last_ingested_at
FROM workspace.bronze.bronze_orders_csv;

total_rows_loaded,distinct_source_files,first_ingested_at,last_ingested_at
9310000,7,2026-05-06T11:21:00.956Z,2026-05-06T11:21:00.956Z


In [0]:
-- STEP 2.3: Ingest 2 JSON files into bronze_orders_json
-- Purpose: Load both JSON files from the Volume into the Bronze JSON table.
--          The JSON files are in ARRAY format (one big [{...}, {...}, ...] rather than line-delimited), which requires the multiLine option.
--
-- Why explicit casting (same approach as CSV)?
--      JSON readers can also infer types loosely (e.g., decimals as strings)
--      Explicit CAST guarantees the data lands in the table's declared types
--      Matches the pattern used in Step 2.3 for consistency
--
-- FORMAT_OPTIONS:
--      multiLine = 'true'      -> required for JSON array format files

COPY INTO workspace.bronze.bronze_orders_json
FROM (
    SELECT
        CAST(order_id       AS BIGINT)  AS order_id,
        CAST(order_date     AS STRING)  AS order_date,
        CAST(hour           AS INT)     AS hour,
        CAST(category       AS STRING)  AS category,
        CAST(item_name      AS STRING)  AS item_name,
        CAST(price          AS DOUBLE)  AS price,
        CAST(quantity       AS INT)     AS quantity,
        CAST(discount       AS DOUBLE)  AS discount,
        CAST(total_amount   AS DOUBLE)  AS total_amount,
        CAST(branch         AS STRING)  AS branch,
        CAST(payment_method AS STRING)  AS payment_method,
        CAST(order_type     AS STRING)  AS order_type,
        CAST(customer_id    AS BIGINT)  AS customer_id,
        CAST(rating         AS INT)     AS rating,
        CAST(is_weekend     AS INT)     AS is_weekend,
        CURRENT_TIMESTAMP()             AS _ingested_at,
        _metadata.file_name             AS _source_file
    FROM '/Volumes/workspace/bronze/raw_files/json/'
)
FILEFORMAT = JSON
FORMAT_OPTIONS (
    'multiLine' = 'true'
);

-- Validate ingestion: count rows loaded into Bronze JSON table
-- Expected: 1,800,000 rows (sum of both JSONs from inspection)
SELECT 
    COUNT(*) AS total_rows_loaded,
    COUNT(DISTINCT _source_file) AS distinct_source_files,
    MIN(_ingested_at) AS first_ingested_at,
    MAX(_ingested_at) AS last_ingested_at
FROM workspace.bronze.bronze_orders_json;

total_rows_loaded,distinct_source_files,first_ingested_at,last_ingested_at
1800000,2,2026-05-06T11:25:01.051Z,2026-05-06T11:25:01.051Z


# Step 3 — Data Quality Report
Before transforming any data, we **profile the Bronze layer** to identify quality issues. The findings will be saved as a permanent view (`bronze.data_quality_report`): a single source of truth that documents the state of the source data.

### Why profile before cleaning?
- **Discover problems** that may not be obvious from a casual look at the data
- **Document findings** so the cleaning logic in Silver has clear justification
- **Establish a baseline** anyone running this report later can compare current quality against today's snapshot
- **Engineering rigor** real production pipelines always profile before transforming

### Checks performed
| # | Check | What it catches |
|---|---|---|
| 1 | Total row counts per source | Sanity check — ensures no missing files |
| 2 | Null check on critical columns | Identifies incomplete records |
| 3 | Duplicate `order_id` count | Reveals upstream system bugs |
| 4 | Negative or zero values in financial fields | Likely data-entry errors |
| 5 | `total_amount` vs `price × quantity × (1 − discount)` | Catches calculation bugs in source |
| 6 | Distinct `branch` values (case sensitivity) | Reveals casing typos like "Cairo" vs "cairo" |
| 7 | Rating values outside 1–5 | Invalid scale entries |
| 8 | `is_weekend` values that aren't 0 or 1 | Boolean field violations |
| 9 | Distinct `order_type` values | Documents the new column not in the original demo |
| 10 | Date range and outliers | Catches future dates or impossibly old dates |

### Key finding (preview)
The headline finding from this report is that **~29.1% of `total_amount` values in source data do not match the formula `price × quantity × (1 − discount)`**. The Silver layer will recompute and correct this column in Step 4, while preserving the original value for full audit traceability.

In [0]:
-- STEP 3.1: Create a Bronze "all sources" view for DQ profiling
-- Purpose: Create a lightweight VIEW that unions the CSV and JSON Bronze tables, so the Data Quality Report can profile the combined raw data in one query.
--           This is NOT the Silver layer: no cleaning happens here. 
--           It is purely a profiling helper.
--
-- Why a VIEW instead of a TABLE?
--      No data duplication (the view is just a query definition)
--      Always reflects the current state of Bronze tables
--      Drops automatically if Bronze tables change

CREATE OR REPLACE VIEW workspace.bronze.bronze_orders_all AS
SELECT *, 'csv' AS _source_format FROM workspace.bronze.bronze_orders_csv
UNION ALL
SELECT *, 'json' AS _source_format FROM workspace.bronze.bronze_orders_json;

-- Validate the view returns the expected combined row count
-- Expected: 11,110,000 (9,310,000 CSV + 1,800,000 JSON)
SELECT 
    _source_format,
    COUNT(*) AS row_count
FROM workspace.bronze.bronze_orders_all
GROUP BY _source_format
ORDER BY _source_format;

_source_format,row_count
csv,9310000
json,1800000


In [0]:
-- STEP 3.2: Build the Data Quality Report view
-- Purpose: Profile the Bronze layer comprehensively. 
-- Each check returns one row with a consistent structure (check_id, check_name, severity, affected_rows, affected_pct, description), and all checks are UNION-ed into a single view: workspace.bronze.data_quality_report.
--
-- Severity levels:
--  INFO     -> Informational. No data quality issue, just useful context.
--  WARNING  -> Minor issue. Data is usable but should be cleaned.
--  CRITICAL -> Major issue. Data must be corrected or quarantined in Silver.
--
-- This view becomes a permanent artifact. Anyone running it later can compare
-- current Bronze quality against the snapshot taken on first ingestion.

CREATE OR REPLACE VIEW workspace.bronze.data_quality_report AS

-- CHECK 1: Total row count per source format (sanity check)
SELECT 
    1 AS check_id,
    'Row counts per source' AS check_name,
    'INFO' AS severity,
    COUNT(*) AS affected_rows,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2) AS affected_pct,
    CONCAT('Total rows in Bronze (CSV + JSON combined): ', COUNT(*)) AS description
FROM workspace.bronze.bronze_orders_all

UNION ALL

-- CHECK 2: Null values in critical columns
-- Why: Critical columns must not be null. Any null here indicates corrupt data.
SELECT 
    2,
    'Null order_id (CRITICAL identifier)',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'CRITICAL' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Rows with null order_id — these cannot be processed and will be quarantined in Silver'
FROM workspace.bronze.bronze_orders_all
WHERE order_id IS NULL

UNION ALL

SELECT 
    3,
    'Null customer_id',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'WARNING' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Rows with null customer_id — would prevent customer-level analytics'
FROM workspace.bronze.bronze_orders_all
WHERE customer_id IS NULL

UNION ALL

SELECT 
    4,
    'Null branch',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'WARNING' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Rows with null branch — would prevent geographic analytics'
FROM workspace.bronze.bronze_orders_all
WHERE branch IS NULL

UNION ALL

-- CHECK 5: Duplicate order_id values
-- Why: order_id is meant to be a unique business key per transaction.
-- Note: this counts the duplicate-occurrences (everything beyond the first).
SELECT 
    5,
    'Duplicate order_id values',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'WARNING' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Order IDs appearing more than once across sources — Silver layer will deduplicate'
FROM (
    SELECT order_id, COUNT(*) AS occurrences
    FROM workspace.bronze.bronze_orders_all
    GROUP BY order_id
    HAVING COUNT(*) > 1
)

UNION ALL

-- CHECK 6: Negative or zero values in financial fields
SELECT 
    6,
    'Invalid price (<= 0)',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'WARNING' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Rows with non-positive price — likely data-entry errors'
FROM workspace.bronze.bronze_orders_all
WHERE price <= 0

UNION ALL

SELECT 
    7,
    'Invalid quantity (<= 0)',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'WARNING' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Rows with non-positive quantity — likely data-entry errors'
FROM workspace.bronze.bronze_orders_all
WHERE quantity <= 0

UNION ALL

SELECT 
    8,
    'Invalid total_amount (<= 0)',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'WARNING' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Rows with non-positive total_amount — likely calculation or data-entry errors'
FROM workspace.bronze.bronze_orders_all
WHERE total_amount <= 0

UNION ALL

-- CHECK 9: HEADLINE FINDING — total_amount mismatches
-- Compares the source total_amount against the recalculated value.
-- A row is flagged as a mismatch if |source - calculated| > 0.01 (penny tolerance).
SELECT 
    9,
    'total_amount mismatch (HEADLINE FINDING)',
    'CRITICAL',
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Source total_amount does not equal price * quantity * (1 - discount). Silver layer will recompute and preserve the original value.'
FROM workspace.bronze.bronze_orders_all
WHERE ABS(total_amount - (price * quantity * (1 - COALESCE(discount, 0)))) > 0.01

UNION ALL

-- CHECK 10: Distinct branches (catches casing typos)
SELECT 
    10,
    'Distinct branch values',
    'INFO',
    COUNT(DISTINCT branch),
    NULL,
    CONCAT('Distinct branches found: ', CONCAT_WS(', ', COLLECT_SET(branch)))
FROM workspace.bronze.bronze_orders_all

UNION ALL

-- CHECK 11: Distinct order_type values
SELECT 
    11,
    'Distinct order_type values',
    'INFO',
    COUNT(DISTINCT order_type),
    NULL,
    CONCAT('Distinct order types found: ', CONCAT_WS(', ', COLLECT_SET(order_type)))
FROM workspace.bronze.bronze_orders_all

UNION ALL

-- CHECK 12: Distinct payment_method values
SELECT 
    12,
    'Distinct payment_method values',
    'INFO',
    COUNT(DISTINCT payment_method),
    NULL,
    CONCAT('Distinct payment methods found: ', CONCAT_WS(', ', COLLECT_SET(payment_method)))
FROM workspace.bronze.bronze_orders_all

UNION ALL

-- CHECK 13: Rating values outside valid 1-5 range
SELECT 
    13,
    'Rating outside 1-5 range',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'WARNING' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Rows with rating < 1 or rating > 5 — invalid scale, will be set to NULL in Silver'
FROM workspace.bronze.bronze_orders_all
WHERE rating < 1 OR rating > 5

UNION ALL

-- CHECK 14: is_weekend values that aren't 0 or 1
SELECT 
    14,
    'Invalid is_weekend values',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'CRITICAL' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Rows where is_weekend is not 0 or 1 — boolean field violation'
FROM workspace.bronze.bronze_orders_all
WHERE is_weekend NOT IN (0, 1)

UNION ALL

-- CHECK 15: Date range sanity check
SELECT 
    15,
    'Date range coverage',
    'INFO',
    COUNT(DISTINCT TO_DATE(order_date)),
    NULL,
    CONCAT('Order dates span from ', MIN(order_date), ' to ', MAX(order_date), ' (', COUNT(DISTINCT TO_DATE(order_date)), ' distinct days)')
FROM workspace.bronze.bronze_orders_all

UNION ALL

-- CHECK 16: Future-dated orders (impossible)
SELECT 
    16,
    'Future-dated orders',
    CASE WHEN COUNT(*) = 0 THEN 'INFO' ELSE 'WARNING' END,
    COUNT(*),
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all), 2),
    'Rows with order_date after today — data-entry errors'
FROM workspace.bronze.bronze_orders_all
WHERE TO_DATE(order_date) > CURRENT_DATE()

ORDER BY check_id;

-- Display the full Data Quality Report
SELECT * FROM workspace.bronze.data_quality_report;

check_id,check_name,severity,affected_rows,affected_pct,description
1,Row counts per source,INFO,11110000,100.00,Total rows in Bronze (CSV + JSON combined): 11110000
2,Null order_id (CRITICAL identifier),INFO,0,0.00,Rows with null order_id — these cannot be processed and will be quarantined in Silver
3,Null customer_id,INFO,0,0.00,Rows with null customer_id — would prevent customer-level analytics
4,Null branch,INFO,0,0.00,Rows with null branch — would prevent geographic analytics
5,Duplicate order_id values,WARNING,2000000,18.00,Order IDs appearing more than once across sources — Silver layer will deduplicate
6,Invalid price (<= 0),WARNING,51442,0.46,Rows with non-positive price — likely data-entry errors
7,Invalid quantity (<= 0),INFO,0,0.00,Rows with non-positive quantity — likely data-entry errors
8,Invalid total_amount (<= 0),WARNING,51409,0.46,Rows with non-positive total_amount — likely calculation or data-entry errors
9,total_amount mismatch (HEADLINE FINDING),CRITICAL,3229776,29.07,Source total_amount does not equal price * quantity * (1 - discount). Silver layer will recompute and preserve the original value.
10,Distinct branch values,INFO,6,null,"Distinct branches found: القاهرة, الجيزة, أسيوط, الإسكندرية, طنطا, المنصورة"


In [0]:
-- Investigation: Are duplicate order_ids genuinely the same order, or do they represent different orders from different source files?
-- This determines whether we should DEDUPLICATE or generate a NEW SURROGATE KEY in the Silver layer.
--
-- Hypothesis: Each source file is a separate branch's POS export, so order_id=1 in restaurant1.csv is a different order than order_id=1 in restaurant2.csv. If so, the rows will have different branch, customer_id, total_amount, etc.

-- Pick a few sample order_ids that appear in multiple files and inspect them
SELECT 
    order_id,
    _source_file,
    order_date,
    hour,
    branch,
    item_name,
    customer_id,
    total_amount
FROM workspace.bronze.bronze_orders_all
WHERE order_id IN (1, 100, 1000)
ORDER BY order_id, _source_file
LIMIT 30;

order_id,_source_file,order_date,hour,branch,item_name,customer_id,total_amount
1,restaurant1.csv,2023-10-28,10,الجيزة,طحينة,168213,34.59
1,restaurant2.csv,2020-11-29,13,الإسكندرية,محشي ورق عنب,79167,494.56
1,restaurant3.csv,2024-10-24,14,القاهرة,طحينة,7834,160.73
1,restaurant4.csv,2023-03-08,19,القاهرة,محشي ورق عنب,21788,515.65
1,restaurant5.csv,2020-06-13,16,القاهرة,بابا غنوج,6274,140.96
1,restaurant6.csv,2025-03-12,20,المنصورة,طاجن بامية,187346,409.06
1,restaurant7.csv,2022-12-18,17,القاهرة,عصير مانجو,196782,81.71
1,restaurant_json1.json,2021-09-25,18,القاهرة,عصير مانجو,2499,104.5
1,restaurant_json2.json,2021-01-05,23,المنصورة,كفتة,104552,697.21
100,restaurant1.csv,2024-01-30,11,الإسكندرية,شيش طاووق,197100,576.78


# Step 4: Silver Layer: Cleaning + Quarantine
The Silver layer transforms raw Bronze data into a **clean, unified, validated** dataset ready for analytical modeling.
Two outputs are produced:
| Table | Purpose |
|---|---|
| `silver.orders_clean` | Cleaned, deduplicated, type-cast, and validated transactions |
| `silver.orders_quarantine` | Invalid rows preserved for audit (never silently dropped) |

### Findings from the Data Quality Report addressed in this step
| Finding | Severity | Resolution |
|---|---|---|
| 18% duplicate `order_id` across source files | WARNING | Generate `unique_order_id` by combining source file + original ID |
| 29.07% `total_amount` mismatches | CRITICAL | Recompute as `price × quantity × (1 − discount)`; preserve original in `total_amount_source`; flag with `total_amount_was_corrected` |
| ~51K rows with invalid price or total_amount | WARNING | Quarantine for audit (do not delete) |
| Casing/whitespace inconsistencies (preventive) | (preventive) | Apply `TRIM()` to text fields |
| `order_date` stored as STRING in Bronze | (preventive) | Cast to proper `DATE` type |

### Why a Quarantine table?
Real production pipelines **never silently drop bad data**.
Quarantining preserves rows that fail validation rules so they can be:
- Audited later to investigate root causes
- Re-processed if the underlying issue is fixed
- Counted in compliance reports
- This is a pattern used at banks, healthcare systems, and any regulated industry. It is the single biggest engineering differentiator in this project.

### Surrogate key strategy
Investigation in Step 3 revealed that duplicate `order_id` values represent **different transactions from different POS exports**, not redundant duplicates. Therefore Silver generates a globally-unique surrogate key:

In [0]:
-- STEP 4.1: Build silver.orders_clean
-- Purpose: Transform the raw Bronze data into a clean, unified, validated Silver table. 
-- This is the single source of truth for the project's downstream analytics.
--
-- Transformations applied:
--  1. Generate surrogate key   -> unique_order_id = source_file + '_' + order_id
--  2. Preserve original key    -> source_order_id (for traceability)
--  3. Cast order_date          -> from STRING to DATE (clean type)
--  4. Trim text fields         -> branch, category, item_name, payment_method, order_type
--  5. Coalesce nulls           -> discount null -> 0 (treat missing as no discount)
--  6. Recompute total_amount   -> from price * quantity * (1 - discount)
--  7. Preserve original total  -> total_amount_source (audit trail)
--  8. Flag corrected rows      -> total_amount_was_corrected (BOOLEAN)
--
-- Validation rules (rows failing these are excluded; quarantined in Step 4.3):
--  order_id must be NOT NULL
--  price must be > 0
--  quantity must be > 0
--  is_weekend must be 0 or 1
--  rating must be between 1 and 5 (inclusive)

CREATE OR REPLACE TABLE workspace.silver.orders_clean
USING DELTA
COMMENT 'Silver layer: cleaned, deduplicated, type-cast, and validated orders. Single source of truth for analytics.'
AS
SELECT
    -- Surrogate key generation
    -- New globally-unique key: combines source filename + original order_id.
    -- Resolves the 18% cross-source order_id collision found in DQ Check 5.
    CONCAT(_source_file, '_', CAST(order_id AS STRING)) AS unique_order_id,
    
    -- Preserve the original order_id from source for traceability/audit
    order_id AS source_order_id,
    
    -- Date and time
    CAST(order_date AS DATE) AS order_date,
    hour,
    
    -- Item and category (text fields trimmed for casing/whitespace consistency)
    TRIM(category)   AS category,
    TRIM(item_name)  AS item_name,
    
    -- Financial fields
    price,
    quantity,
    COALESCE(discount, 0) AS discount, -- null discount means "no discount applied"
    
    -- HEADLINE FIX: recomputed total_amount (corrects the 29.07% mismatch)
    ROUND(price * quantity * (1 - COALESCE(discount, 0)), 2) AS total_amount,
    
    -- Audit trail: original total_amount from source (may differ from recomputed)
    total_amount AS total_amount_source,
    
    -- Boolean flag: TRUE if the row's total_amount was corrected by Silver
    CASE 
        WHEN ABS(total_amount - (price * quantity * (1 - COALESCE(discount, 0)))) > 0.01 
        THEN TRUE 
        ELSE FALSE 
    END AS total_amount_was_corrected,
    
    -- Categorical fields (trimmed)
    TRIM(branch)         AS branch,
    TRIM(payment_method) AS payment_method,
    TRIM(order_type)     AS order_type,
    
    -- Customer and rating
    customer_id,
    rating,
    is_weekend,
    
    -- Lineage metadata (carried forward from Bronze)
    _source_file,
    _source_format,
    _ingested_at
    
FROM workspace.bronze.bronze_orders_all
WHERE 
    -- Validation rules: rows passing ALL of these land in orders_clean.
    order_id IS NOT NULL
    AND price > 0
    AND quantity > 0
    AND is_weekend IN (0, 1)
    AND rating BETWEEN 1 AND 5;

-- Validate the Silver build
SELECT
    COUNT(*)                                                AS total_rows_clean,
    COUNT(DISTINCT unique_order_id)                         AS distinct_unique_keys,
    SUM(CASE WHEN total_amount_was_corrected THEN 1 ELSE 0 END) AS rows_corrected,
    ROUND(100.0 * SUM(CASE WHEN total_amount_was_corrected THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_corrected,
    MIN(order_date) AS earliest_order,
    MAX(order_date) AS latest_order
FROM workspace.silver.orders_clean;

total_rows_clean,distinct_unique_keys,rows_corrected,pct_corrected,earliest_order,latest_order
11058558,11058558,3213748,29.06,2020-01-01,2025-12-30


In [0]:
-- STEP 4.2: Build silver.orders_quarantine
-- Purpose: Preserve every row that failed Silver validation. 
--          Each row is tagged with the specific quarantine reason for audit and potential re-processing.
--
-- Why quarantine instead of delete?
--  Real production pipelines NEVER silently drop bad data
--  Quarantined rows can be audited to investigate root causes
--  If a source-system bug is later fixed upstream, rows can be re-processed
--  Compliance/regulatory frameworks require this pattern
--
-- Quarantine reasons (mutually compatible — a row can fail multiple checks, but is tagged with the FIRST applicable reason for clarity):
--   1. Null order_id
--   2. Non-positive price
--   3. Non-positive quantity
--   4. Invalid is_weekend value
--   5. Invalid rating value

CREATE OR REPLACE TABLE workspace.silver.orders_quarantine
USING DELTA
COMMENT 'Silver layer: rows that failed validation rules. Preserved for audit and potential re-processing. Never silently dropped.'
AS
SELECT
    -- All original Bronze columns (preserved as-is for forensic analysis)
    order_id,
    order_date,
    hour,
    category,
    item_name,
    price,
    quantity,
    discount,
    total_amount,
    branch,
    payment_method,
    order_type,
    customer_id,
    rating,
    is_weekend,
    
    -- Quarantine reason: first applicable validation failure
    CASE
        WHEN order_id IS NULL              THEN 'NULL_ORDER_ID'
        WHEN price IS NULL OR price <= 0   THEN 'INVALID_PRICE'
        WHEN quantity IS NULL OR quantity <= 0 THEN 'INVALID_QUANTITY'
        WHEN is_weekend NOT IN (0, 1)      THEN 'INVALID_IS_WEEKEND'
        WHEN rating NOT BETWEEN 1 AND 5    THEN 'INVALID_RATING'
        ELSE 'UNKNOWN'
    END AS quarantine_reason,
    
    -- Lineage metadata (carried forward from Bronze)
    _source_file,
    _source_format,
    _ingested_at,
    CURRENT_TIMESTAMP() AS _quarantined_at
    
FROM workspace.bronze.bronze_orders_all
WHERE 
    -- Inverse of the orders_clean WHERE clause: any row failing ANY rule
    order_id IS NULL
    OR price IS NULL OR price <= 0
    OR quantity IS NULL OR quantity <= 0
    OR is_weekend NOT IN (0, 1)
    OR rating NOT BETWEEN 1 AND 5;

-- Quarantine report: row count by reason
SELECT
    quarantine_reason,
    COUNT(*) AS row_count,
    ROUND(100.0 * COUNT(*) / 
        (SELECT COUNT(*) FROM workspace.silver.orders_quarantine), 2) AS pct_of_quarantine
FROM workspace.silver.orders_quarantine
GROUP BY quarantine_reason
ORDER BY row_count DESC;

quarantine_reason,row_count,pct_of_quarantine
INVALID_PRICE,51442,100.00


In [0]:
-- Conservation check: clean + quarantine should equal Bronze total
-- This verifies that NO rows were silently lost between Bronze and Silver.
-- Bronze rows == clean rows + quarantine rows

SELECT
    (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all)   AS bronze_total,
    (SELECT COUNT(*) FROM workspace.silver.orders_clean)        AS silver_clean,
    (SELECT COUNT(*) FROM workspace.silver.orders_quarantine)   AS silver_quarantine,
    (SELECT COUNT(*) FROM workspace.silver.orders_clean) +
    (SELECT COUNT(*) FROM workspace.silver.orders_quarantine)   AS clean_plus_quarantine,
    CASE 
        WHEN (SELECT COUNT(*) FROM workspace.bronze.bronze_orders_all)
           = (SELECT COUNT(*) FROM workspace.silver.orders_clean) +
             (SELECT COUNT(*) FROM workspace.silver.orders_quarantine)
        THEN 'PASS: All Bronze rows accounted for'
        ELSE 'FAIL: Row count mismatch — investigate'
    END AS conservation_check;

bronze_total,silver_clean,silver_quarantine,clean_plus_quarantine,conservation_check
11110000,11058558,51442,11110000,PASS: All Bronze rows accounted for


# Step 5: Silver Layer: Time Enrichment

Three derived time columns are added to `silver.orders_clean` to support time-based dashboard analytics:
| Column | Type | Source | Used by |
|---|---|---|---|
| `year` | INT | `YEAR(order_date)` | Revenue trend chart, Year filter context |
| `month` | INT | `MONTH(order_date)` | Revenue trend chart (monthly granularity) |
| `day_name` | STRING | `DATE_FORMAT(order_date, 'EEEE')` | Hourly × Day-of-Week heatmap |

### Why enrich in Silver instead of Power BI?
- **Performance** Power BI reads pre-computed values; no on-the-fly calculation
- **Reusability** these columns will populate `dim_date` in the Gold layer
- **Consistency** every consumer of `orders_clean` sees the same enrichment

In [0]:
-- STEP 5: Add time enrichment columns to silver.orders_clean
-- Purpose: Add year, month, and day_name columns derived from order_date.
--          These columns are computed once and reused by every downstream consumer (dim_date, KPI views, Power BI charts).
--
-- Approach: ALTER TABLE adds the columns as nullable; UPDATE populates them in a single pass.
--           Delta Lake handles this as an atomic operation.

-- Step 5.a: Add the three new columns (initially NULL)
ALTER TABLE workspace.silver.orders_clean 
ADD COLUMNS (
    year     INT     COMMENT 'Calendar year extracted from order_date',
    month    INT     COMMENT 'Calendar month (1-12) extracted from order_date',
    day_name STRING  COMMENT 'Full day name in English (e.g., Monday, Tuesday)'
);

-- Step 5.b: Populate the three columns with derived values
UPDATE workspace.silver.orders_clean
SET 
    year     = YEAR(order_date),
    month    = MONTH(order_date),
    day_name = DATE_FORMAT(order_date, 'EEEE');

-- Step 5.c: Validate the enrichment worked
SELECT 
    COUNT(*)                    AS total_rows,
    COUNT(year)                 AS year_populated,
    COUNT(month)                AS month_populated,
    COUNT(day_name)             AS day_name_populated,
    MIN(year)                   AS earliest_year,
    MAX(year)                   AS latest_year,
    COUNT(DISTINCT day_name)    AS distinct_day_names
FROM workspace.silver.orders_clean;

total_rows,year_populated,month_populated,day_name_populated,earliest_year,latest_year,distinct_day_names
11058558,11058558,11058558,11058558,2020,2025,7


# Step 6: Gold Layer: Star Schema
The Gold layer restructures Silver data into a **dimensional model** optimized for analytical queries and BI tools. The schema follows the classic **star pattern**: one central fact table surrounded by descriptive dimension tables, joined via integer surrogate keys.

### Tables built in this step
| Table | Type | Grain | Surrogate Key |
|---|---|---|---|
| `gold.dim_date` | Dimension | One row per calendar date | `date_key` |
| `gold.dim_branch` | Dimension | One row per branch | `branch_key` |
| `gold.dim_item` | Dimension | One row per menu item | `item_key` |
| `gold.dim_payment` | Dimension | One row per payment method | `payment_key` |
| `gold.fact_orders` | Fact | One row per transaction | `unique_order_id` |

### Why a star schema?
| Reason | Benefit |
|---|---|
| **Industry-standard** | Every BI team uses this pattern; clients expect it |
| **Query performance** | INT key joins are 5–10× faster than string joins |
| **Power BI compatible** | Power BI auto-detects relationships and renders the star |
| **Reusable** | New analyses plug into existing dimensions, no rebuild needed |

### Surrogate key strategy
Each dimension generates an INT surrogate key using `ROW_NUMBER() OVER (ORDER BY ...)`. The key is meaningless on its own — it exists purely to make joins fast and consistent. Natural business identifiers are preserved as separate columns.

In [0]:
-- STEP 6.1: Build gold.dim_date
-- Purpose: Date dimension table. 
--          One row per distinct calendar date in the dataset, with all derived date attributes pre-computed for fast slicing in Power BI.
--
-- Grain: One row per calendar date.
-- Surrogate key: date_key (INT, generated via ROW_NUMBER)

CREATE OR REPLACE TABLE workspace.gold.dim_date
USING DELTA
COMMENT 'Date dimension. One row per calendar date with pre-computed attributes for time-based analysis.'
AS
SELECT
    -- Surrogate key: deterministic INT generated from ordered distinct dates
    ROW_NUMBER() OVER (ORDER BY order_date) AS date_key,
    
    -- The full date value (the natural key)
    order_date AS full_date,
    
    -- Pre-computed date attributes (same as in silver.orders_clean, repeated here so the dimension is self-contained, as joining to dim_date alone gives you everything you need without going back to the fact table)
    YEAR(order_date)                AS year,
    MONTH(order_date)               AS month,
    DATE_FORMAT(order_date, 'MMMM') AS month_name,
    QUARTER(order_date)             AS quarter,
    DAY(order_date)                 AS day_of_month,
    DATE_FORMAT(order_date, 'EEEE') AS day_name,
    DAYOFWEEK(order_date)           AS day_of_week, -- 1=Sunday, 7=Saturday
    WEEKOFYEAR(order_date)          AS week_of_year,
    
    -- Boolean flag for weekend (computed from day_of_week)
    CASE WHEN DAYOFWEEK(order_date) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend
    
FROM (
    -- Distinct dates only: dim_date should have one row per date, not per order
    SELECT DISTINCT order_date 
    FROM workspace.silver.orders_clean
    WHERE order_date IS NOT NULL
);

-- Validate: row count should equal distinct dates in Silver
SELECT 
    COUNT(*)             AS total_dates,
    MIN(full_date)       AS earliest_date,
    MAX(full_date)       AS latest_date,
    COUNT(DISTINCT year) AS distinct_years,
    MIN(date_key)        AS min_key,
    MAX(date_key)        AS max_key
FROM workspace.gold.dim_date;

total_dates,earliest_date,latest_date,distinct_years,min_key,max_key
2191,2020-01-01,2025-12-30,6,1,2191


In [0]:
-- STEP 6.2: Build gold.dim_branch
-- Purpose: Branch dimension table. One row per distinct branch.
--
-- Grain: One row per branch.
-- Surrogate key: branch_key (INT, generated via ROW_NUMBER)
--
-- Design note: An English translation of the Arabic branch name is added as a separate column. 
--              This allow Power BI dashboard to display either Arabic and English:

CREATE OR REPLACE TABLE workspace.gold.dim_branch
USING DELTA
COMMENT 'Branch dimension. One row per distinct branch, with both Arabic and English names.'
AS
SELECT
    -- Surrogate key
    ROW_NUMBER() OVER (ORDER BY branch) AS branch_key,
    
    -- Natural key (the original Arabic branch name from source data)
    branch AS branch_name_ar,
    
    -- English translation for international audiences and dashboard flexibility
    CASE branch
        WHEN 'القاهرة'     THEN 'Cairo'
        WHEN 'الإسكندرية'   THEN 'Alexandria'
        WHEN 'الجيزة'      THEN 'Giza'
        WHEN 'المنصورة'    THEN 'Mansoura'
        WHEN 'طنطا'        THEN 'Tanta'
        WHEN 'أسيوط'       THEN 'Asyut'
        ELSE branch
    END AS branch_name_en
    
FROM (
    SELECT DISTINCT branch 
    FROM workspace.silver.orders_clean
    WHERE branch IS NOT NULL
);

-- Validate: should show all 6 branches with their English translations
SELECT * 
FROM workspace.gold.dim_branch
ORDER BY branch_key;

branch_key,branch_name_ar,branch_name_en
1,أسيوط,Asyut
2,الإسكندرية,Alexandria
3,الجيزة,Giza
4,القاهرة,Cairo
5,المنصورة,Mansoura
6,طنطا,Tanta


In [0]:
-- STEP 6.3: Build gold.dim_item
-- Purpose: Item dimension table. 
--          One row per distinct (category, item_name) combination found in the data.
--
-- Grain: One row per unique item.
-- Surrogate key: item_key (INT, generated via ROW_NUMBER)
--
-- Design note: We use the (category, item_name) pair as the natural key rather than item_name alone. 
--              This protects against the rare case where the same item name appears under two different categories (e.g., a beverage and a dessert sharing a name).              

CREATE OR REPLACE TABLE workspace.gold.dim_item
USING DELTA
COMMENT 'Item dimension. One row per unique (category, item_name) combination.'
AS
SELECT
    -- Surrogate key
    ROW_NUMBER() OVER (ORDER BY category, item_name) AS item_key,
    
    -- Natural key components
    category,
    item_name
    
FROM (
    SELECT DISTINCT category, item_name 
    FROM workspace.silver.orders_clean
    WHERE category IS NOT NULL 
      AND item_name IS NOT NULL
);

-- Validate: row count, key range, and a sample of items
SELECT
    COUNT(*)                  AS total_items,
    COUNT(DISTINCT category)  AS distinct_categories,
    COUNT(DISTINCT item_name) AS distinct_item_names,
    MIN(item_key)             AS min_key,
    MAX(item_key)             AS max_key
FROM workspace.gold.dim_item;

total_items,distinct_categories,distinct_item_names,min_key,max_key
15,5,15,1,15


In [0]:
-- Show 10 sample items to verify the data looks right
SELECT * FROM workspace.gold.dim_item
ORDER BY item_key
LIMIT 10;

item_key,category,item_name
1,طواجن,طاجن بامية
2,طواجن,طاجن فراخ
3,طواجن,طاجن ملوخية
4,محاشي,محشي باذنجان
5,محاشي,محشي كوسة
6,محاشي,محشي ورق عنب
7,مشروبات,شاي
8,مشروبات,عصير قصب
9,مشروبات,عصير مانجو
10,مشويات,شيش طاووق


In [0]:
-- STEP 6.5: Build gold.dim_payment
-- Purpose: Payment method dimension table. One row per distinct payment method.
--
-- Grain: One row per payment method.
-- Surrogate key: payment_key (INT, generated via ROW_NUMBER)
--
-- From DQ Check 12, we know there are exactly 3 payment methods: Cash, Card, Wallet

CREATE OR REPLACE TABLE workspace.gold.dim_payment
USING DELTA
COMMENT 'Payment method dimension. One row per distinct payment method.'
AS
SELECT
    -- Surrogate key
    ROW_NUMBER() OVER (ORDER BY payment_method) AS payment_key,
    
    -- Natural key
    payment_method
    
FROM (
    SELECT DISTINCT payment_method
    FROM workspace.silver.orders_clean
    WHERE payment_method IS NOT NULL
);

-- Validate: should show all 3 payment methods
SELECT *
FROM workspace.gold.dim_payment
ORDER BY payment_key;

payment_key,payment_method
1,Card
2,Cash
3,Wallet


In [0]:
-- STEP 6.5: Build gold.fact_orders (the central fact table)
-- Purpose: The heart of the star schema. 
--          One row per transaction, with all natural dimension values replaced by surrogate INT keys (foreign keys pointing to dim_date, dim_branch, dim_item, dim_payment).
--
-- Grain: One row per order (using unique_order_id from Silver).
--
-- Foreign keys (resolved via JOIN to each dimension):
--  date_key    -> dim_date
--  branch_key  -> dim_branch
--  item_key    -> dim_item
--  payment_key -> dim_payment
--
-- Degenerate dimensions (kept directly on fact, no separate dim needed):
--  order_type, is_weekend, hour, day_name
--
-- Measures (numeric values for aggregation):
--  price, quantity, discount, total_amount, rating

CREATE OR REPLACE TABLE workspace.gold.fact_orders
USING DELTA
COMMENT 'Fact table at order grain. Foreign-keyed to dim_date, dim_branch, dim_item, dim_payment. Joined via INT surrogate keys for fast analytical queries.'
AS
SELECT
    -- Identifiers (preserved from Silver for traceability)
    s.unique_order_id, -- globally unique key (from Silver)
    s.source_order_id, -- original order_id from source
    s.customer_id,     -- kept as natural key (no dim_customer in this scope)
    
    -- Foreign keys to dimensions (the "star" relationships)
    d.date_key,
    b.branch_key,
    i.item_key,
    p.payment_key,
    
    -- Degenerate dimensions (low-cardinality attributes kept on the fact)
    s.order_type, -- Dine-in / Delivery / Takeaway (3 values)
    s.hour,       -- 0-23
    s.is_weekend, -- 0 or 1
    s.day_name,   -- Monday-Sunday (also in dim_date)
    s.year,       -- duplicated for filter performance
    s.month,      -- duplicated for filter performance
    
    -- Measures (numeric values for aggregation)
    s.price,
    s.quantity,
    s.discount,
    s.total_amount,               -- the CORRECTED value (from Silver)
    s.total_amount_source,        -- original (possibly inaccurate) for audit
    s.total_amount_was_corrected, -- TRUE if Silver corrected this row
    s.rating,
    
    -- Lineage metadata
    s._source_file,
    s._source_format
    
FROM workspace.silver.orders_clean s
INNER JOIN workspace.gold.dim_date    d ON s.order_date     = d.full_date
INNER JOIN workspace.gold.dim_branch  b ON s.branch         = b.branch_name_ar
INNER JOIN workspace.gold.dim_item    i ON s.category       = i.category AND s.item_name = i.item_name
INNER JOIN workspace.gold.dim_payment p ON s.payment_method = p.payment_method;

-- Validation: confirm fact_orders matches Silver row count and key integrity
SELECT
    COUNT(*)                        AS total_fact_rows,
    COUNT(DISTINCT unique_order_id) AS distinct_order_keys,
    COUNT(DISTINCT date_key)        AS distinct_dates_used,
    COUNT(DISTINCT branch_key)      AS distinct_branches_used,
    COUNT(DISTINCT item_key)        AS distinct_items_used,
    COUNT(DISTINCT payment_key)     AS distinct_payments_used,
    SUM(CASE WHEN date_key    IS NULL THEN 1 ELSE 0 END) AS null_date_keys,
    SUM(CASE WHEN branch_key  IS NULL THEN 1 ELSE 0 END) AS null_branch_keys,
    SUM(CASE WHEN item_key    IS NULL THEN 1 ELSE 0 END) AS null_item_keys,
    SUM(CASE WHEN payment_key IS NULL THEN 1 ELSE 0 END) AS null_payment_keys
FROM workspace.gold.fact_orders;

total_fact_rows,distinct_order_keys,distinct_dates_used,distinct_branches_used,distinct_items_used,distinct_payments_used,null_date_keys,null_branch_keys,null_item_keys,null_payment_keys
11058558,11058558,2191,6,15,3,0,0,0,0


# Step 7: Gold Layer: Business KPI Views
Five **pre-aggregated views** are built directly on top of the star schema. 
Each view corresponds to exactly one Power BI visual, ensuring the dashboard loads instantly without recomputing aggregates on every refresh.

### Why pre-aggregated views?
| Reason | Benefit |
|---|---|
| **Performance** | Power BI reads ready-made numbers, not 11M raw rows |
| **Consistency** | Every consumer sees the same KPI logic |
| **Maintainability** | Change the metric definition once, everyone benefits |
| **Storage** | Views take zero storage; they're computed on demand |

### Views built in this step
| View | Powers | Granularity |
|---|---|---|
| `gold.kpi_executive_summary` | The 4 KPI cards (Revenue, Orders, AOV, Rating) | Single row with current and previous period |
| `gold.kpi_monthly_revenue` | Revenue Trend line chart | One row per (year, month) |
| `gold.kpi_top_items` | Top 10 Items bar chart | One row per item |
| `gold.kpi_hourly_heatmap` | Hourly × Day-of-Week heatmap | One row per (day_name, hour) |
| `gold.kpi_payment_mix` | Payment Method donut chart | One row per payment_method |

In [0]:
-- STEP 7.1: Build gold.kpi_executive_summary
-- Purpose: Single-row summary of the 4 headline KPIs PLUS their previous-period values, so Power BI can compute Month-over-Month change indicators.
--
-- Powers: The 4 KPI cards on the dashboard
--  1. Total Revenue
--  2. Total Orders
--  3. Average Order Value
--  4. Average Rating
--
-- Definition of "current" and "previous" period:
--  - current  = the latest month present in the data
--  - previous = the month immediately before the latest
-- This makes the MoM logic dynamic — works for any data refresh.

CREATE OR REPLACE VIEW workspace.gold.kpi_executive_summary AS
WITH 
-- Find the latest month in the data (drives the "current" period)
latest_month AS (
    SELECT 
        MAX(year)  AS curr_year,
        MAX(month) AS curr_month
    FROM workspace.gold.fact_orders
    WHERE year = (SELECT MAX(year) FROM workspace.gold.fact_orders)
),

-- Compute the "previous" month relative to latest
previous_month AS (
    SELECT
        CASE WHEN curr_month = 1 THEN curr_year - 1 ELSE curr_year END AS prev_year,
        CASE WHEN curr_month = 1 THEN 12            ELSE curr_month - 1 END AS prev_month
    FROM latest_month
),

-- Aggregate metrics for the current month
current_metrics AS (
    SELECT
        SUM(total_amount) AS revenue,
        COUNT(*)          AS orders,
        AVG(total_amount) AS avg_order_value,
        AVG(rating)       AS avg_rating
    FROM workspace.gold.fact_orders
    WHERE year  = (SELECT curr_year  FROM latest_month)
      AND month = (SELECT curr_month FROM latest_month)
),

-- Aggregate metrics for the previous month
previous_metrics AS (
    SELECT
        SUM(total_amount) AS revenue,
        COUNT(*)          AS orders,
        AVG(total_amount) AS avg_order_value,
        AVG(rating)       AS avg_rating
    FROM workspace.gold.fact_orders
    WHERE year  = (SELECT prev_year  FROM previous_month)
      AND month = (SELECT prev_month FROM previous_month)
),

-- Lifetime totals (for context cards if needed)
lifetime_metrics AS (
    SELECT
        SUM(total_amount) AS revenue_lifetime,
        COUNT(*)          AS orders_lifetime,
        AVG(total_amount) AS avg_order_value_lifetime,
        AVG(rating)       AS avg_rating_lifetime
    FROM workspace.gold.fact_orders
)

SELECT
    -- Current month metrics (used by the 4 cards)
    ROUND(c.revenue,         2) AS current_revenue,
    c.orders                    AS current_orders,
    ROUND(c.avg_order_value, 2) AS current_avg_order_value,
    ROUND(c.avg_rating,      2) AS current_avg_rating,
    
    -- Previous month metrics (used by Power BI to compute MoM ▲▼)
    ROUND(p.revenue,         2) AS previous_revenue,
    p.orders                    AS previous_orders,
    ROUND(p.avg_order_value, 2) AS previous_avg_order_value,
    ROUND(p.avg_rating,      2) AS previous_avg_rating,
    
    -- Pre-computed MoM percentage changes (Power BI can also compute these)
    ROUND(100.0 * (c.revenue         - p.revenue)         / NULLIF(p.revenue, 0),         2) AS mom_revenue_pct,
    ROUND(100.0 * (c.orders          - p.orders)          / NULLIF(p.orders, 0),          2) AS mom_orders_pct,
    ROUND(100.0 * (c.avg_order_value - p.avg_order_value) / NULLIF(p.avg_order_value, 0), 2) AS mom_aov_pct,
    ROUND(100.0 * (c.avg_rating      - p.avg_rating)      / NULLIF(p.avg_rating, 0),      2) AS mom_rating_pct,
    
    -- Lifetime context (for tooltips or alternative card configurations)
    ROUND(l.revenue_lifetime,         2) AS lifetime_revenue,
    l.orders_lifetime                    AS lifetime_orders,
    ROUND(l.avg_order_value_lifetime, 2) AS lifetime_avg_order_value,
    ROUND(l.avg_rating_lifetime,      2) AS lifetime_avg_rating,
    
    -- Period labels (so Power BI can display "December 2025 vs November 2025")
    (SELECT curr_year  FROM latest_month)   AS current_year,
    (SELECT curr_month FROM latest_month)   AS current_month,
    (SELECT prev_year  FROM previous_month) AS previous_year,
    (SELECT prev_month FROM previous_month) AS previous_month
FROM current_metrics  c
CROSS JOIN previous_metrics p
CROSS JOIN lifetime_metrics l;

-- Validate: should return exactly 1 row
SELECT * FROM workspace.gold.kpi_executive_summary;

current_revenue,current_orders,current_avg_order_value,current_avg_rating,previous_revenue,previous_orders,previous_avg_order_value,previous_avg_rating,mom_revenue_pct,mom_orders_pct,mom_aov_pct,mom_rating_pct,lifetime_revenue,lifetime_orders,lifetime_avg_order_value,lifetime_avg_rating,current_year,current_month,previous_year,previous_month
3.924699504E7,151374,259.27,3.71,4.02002318E7,151778,264.86,3.71,-2.37,-0.27,-2.11,0.05,2.89914705219E9,11058558,262.16,3.7,2025,12,2025,11


In [0]:
-- STEP 7.2: Build gold.kpi_monthly_revenue
-- Purpose: Time-series aggregation of revenue and orders by month. 
--          Powers the Revenue Trend line chart on the dashboard.
--
-- Grain: One row per (year, month) combination.
-- Expected rows: 72 (6 years x 12 months): though some boundary months may be partial depending on the exact date range.                

CREATE OR REPLACE VIEW workspace.gold.kpi_monthly_revenue AS
SELECT
    f.year,
    f.month,
    
    -- Build a month-start date for time-series plotting in Power BI
    -- (Power BI plots dates more cleanly than separate year/month columns)
    MAKE_DATE(f.year, f.month, 1)                AS month_start_date,
    
    -- Display-friendly month label (e.g., "Dec 2025")
    DATE_FORMAT(MAKE_DATE(f.year, f.month, 1), 'MMM yyyy') AS month_label,
    
    -- Core KPIs
    ROUND(SUM(f.total_amount), 2)                AS revenue,
    COUNT(*)                                     AS order_count,
    ROUND(AVG(f.total_amount), 2) AS avg_order_value,
    SUM(f.quantity)                              AS total_quantity,
    ROUND(AVG(f.rating), 2)                      AS avg_rating
    
FROM workspace.gold.fact_orders f
GROUP BY f.year, f.month
ORDER BY f.year, f.month;

-- Validate: show all months and confirm row count
SELECT 
    COUNT(*)                       AS total_months,
    MIN(month_start_date)          AS earliest_month,
    MAX(month_start_date)          AS latest_month,
    ROUND(SUM(revenue), 2)         AS total_lifetime_revenue,
    SUM(order_count)               AS total_lifetime_orders
FROM workspace.gold.kpi_monthly_revenue;

total_months,earliest_month,latest_month,total_lifetime_revenue,total_lifetime_orders
72,2020-01-01,2025-12-01,2.89914705219E9,11058558


In [0]:
-- STEP 7.3: Build gold.kpi_top_items
-- Purpose: Per-item revenue and volume aggregation. 
--          Powers the Top 10 Items bar chart on the dashboard.
--
-- Grain: One row per (category, item_name): same as dim_item.
-- Expected rows: 15 (one per menu item).
--
-- The view returns ALL items, sorted by revenue. 
-- Power BI applies its own TOP N filter (typically TOP 10) so the user can change it interactively.

CREATE OR REPLACE VIEW workspace.gold.kpi_top_items AS
SELECT
    -- Foreign key (links to dim_item if Power BI wants to use the star)
    i.item_key,
    
    -- Descriptive attributes
    i.category,
    i.item_name,
    
    -- Aggregated measures
    ROUND(SUM(f.total_amount), 2)         AS revenue,
    COUNT(*)                              AS order_count,
    SUM(f.quantity)                       AS total_quantity,
    ROUND(AVG(f.total_amount), 2)         AS avg_order_value,
    ROUND(AVG(f.rating), 2)               AS avg_rating,
    
    -- Revenue rank (handy if Power BI wants to label "rank #1, #2, ..." badges)
    DENSE_RANK() OVER (ORDER BY SUM(f.total_amount) DESC) AS revenue_rank,
    
    -- Share of total revenue (useful for percentage labels in tooltips)
    ROUND(100.0 * SUM(f.total_amount) / 
        SUM(SUM(f.total_amount)) OVER (), 2) AS pct_of_total_revenue
    
FROM workspace.gold.fact_orders f
INNER JOIN workspace.gold.dim_item i ON f.item_key = i.item_key
GROUP BY i.item_key, i.category, i.item_name
ORDER BY revenue DESC;

-- Validate: show all 15 items ranked by revenue
SELECT * 
FROM workspace.gold.kpi_top_items
ORDER BY revenue_rank;

item_key,category,item_name,revenue,order_count,total_quantity,avg_order_value,avg_rating,revenue_rank,pct_of_total_revenue
11,مشويات,كباب,5.0528830478E8,1110037,3559729,455.2,3.71,1,17.43
2,طواجن,طاجن فراخ,3.2331204718E8,888354,2846846,363.95,3.7,2,11.15
12,مشويات,كفتة,3.040890393E8,668344,2141993,454.99,3.71,3,10.49
3,طواجن,طاجن ملوخية,2.4287404771E8,667036,2138811,364.11,3.7,4,8.38
1,طواجن,طاجن بامية,2.4261089433E8,666441,2136595,364.04,3.7,5,8.37
6,محاشي,محشي ورق عنب,2.1562616004E8,888435,2847948,242.7,3.7,6,7.44
5,محاشي,محشي كوسة,2.153781051E8,888235,2845125,242.48,3.71,7,7.43
10,مشويات,شيش طاووق,2.0193270859E8,443870,1422770,454.94,3.71,8,6.97
15,مقبلات,طحينة,1.6856246808E8,1110699,3560726,151.76,3.7,9,5.81
9,مشروبات,عصير مانجو,1.2204918682E8,1305573,4185158,93.48,3.7,10,4.21


In [0]:
-- STEP 7.4: Build gold.kpi_hourly_heatmap
-- Purpose: Two-dimensional aggregation by day-of-week and hour-of-day.
--          Powers the Hourly x Day-of-Week heatmap chart, which reveals when the business is busiest: directly actionable for staffing and marketing decisions.
--
-- Grain: One row per (day_name, hour) combination.
-- Expected rows: 7 days x 24 hours = 168 rows
--
-- The heatmap is the most insight-dense visual on the dashboard. 
-- It instantly shows: peak hours, slow periods, weekday vs weekend patterns, and staffing windows.

CREATE OR REPLACE VIEW workspace.gold.kpi_hourly_heatmap AS
SELECT
    -- Day of the week (English name)
    f.day_name,
    
    -- Numeric day order
    CASE f.day_name
        WHEN 'Saturday'  THEN 1
        WHEN 'Sunday'    THEN 2
        WHEN 'Monday'    THEN 3
        WHEN 'Tuesday'   THEN 4
        WHEN 'Wednesday' THEN 5
        WHEN 'Thursday'  THEN 6
        WHEN 'Friday'    THEN 7
    END AS day_sort_order,
    
    CASE 
        WHEN f.day_name IN ('Friday', 'Saturday') THEN 1 
        ELSE 0 
    END AS is_weekend_local,
    
    -- Hour of day (0-23 in source; populated range is 10-23)
    f.hour,
    
    -- Volume metric (typical heatmap color encoding)
    COUNT(*)                            AS order_count,
    
    -- Revenue metric (alternative color encoding)
    ROUND(SUM(f.total_amount), 2)       AS revenue,
    
    -- Average order value at that time slot
    ROUND(AVG(f.total_amount), 2)       AS avg_order_value
    
FROM workspace.gold.fact_orders f
GROUP BY f.day_name, f.hour
ORDER BY day_sort_order, f.hour;

-- Validate: confirm 98 rows (7 days x 14 operating hours), inspect day ordering
SELECT 
    day_name,
    day_sort_order,
    is_weekend_local,
    SUM(order_count)  AS total_orders,
    ROUND(SUM(revenue), 2) AS total_revenue
FROM workspace.gold.kpi_hourly_heatmap
GROUP BY day_name, day_sort_order, is_weekend_local
ORDER BY day_sort_order;

day_name,day_sort_order,is_weekend_local,total_orders,total_revenue
Saturday,1,1,1580730,4.8300721369E8
Sunday,2,0,1579709,4.8323238196E8
Monday,3,0,1580122,3.6258220396E8
Tuesday,4,0,1577863,3.6231498388E8
Wednesday,5,0,1580371,3.6282151916E8
Thursday,6,0,1580465,3.6277561604E8
Friday,7,1,1579298,4.824131335E8


In [0]:
-- STEP 7.5: Build gold.kpi_payment_mix
-- Purpose: Per-payment-method aggregation for the dashboard's donut chart.
--          Shows the share of revenue captured by each payment method (Cash, Card, Wallet) across the entire dataset.
--
-- Grain: One row per payment method.
-- Expected rows: 3 (Card, Cash, Wallet: confirmed by DQ Check 12).

CREATE OR REPLACE VIEW workspace.gold.kpi_payment_mix AS
SELECT
    -- Foreign key (links to dim_payment if Power BI wants the star)
    p.payment_key,
    
    -- Descriptive attribute
    p.payment_method,
    
    -- Volume metrics
    COUNT(*)        AS order_count,
    SUM(f.quantity) AS total_quantity,
    
    -- Revenue metrics
    ROUND(SUM(f.total_amount), 2) AS revenue,
    ROUND(AVG(f.total_amount), 2) AS avg_order_value,
    
    -- Share of total (for donut chart percentage labels)
    ROUND(100.0 * COUNT(*) / 
        SUM(COUNT(*)) OVER (), 2) AS pct_of_orders,
    ROUND(100.0 * SUM(f.total_amount) / 
        SUM(SUM(f.total_amount)) OVER (), 2) AS pct_of_revenue
    
FROM workspace.gold.fact_orders f
INNER JOIN workspace.gold.dim_payment p ON f.payment_key = p.payment_key
GROUP BY p.payment_key, p.payment_method
ORDER BY revenue DESC;

-- Validate: confirm 3 rows and that percentages sum to 100%
SELECT * FROM workspace.gold.kpi_payment_mix;

payment_key,payment_method,order_count,total_quantity,revenue,avg_order_value,pct_of_orders,pct_of_revenue
2,Cash,5530408,17729193,1.4496659354E9,262.13,50.01,50.0
1,Card,3316665,10630989,8.6980453675E8,262.25,29.99,30.0
3,Wallet,2211485,7089226,5.7967658004E8,262.12,20.00,19.99


# Step 8: Performance Optimization (Delta Lake)
The Delta Lake engine provides native commands for physical data layout optimization. Two key operations are applied to the largest tables in this project:
| Command | Effect |
|---|---|
| `OPTIMIZE` | Compacts many small Parquet files into fewer large files (~1 GB target) — reduces I/O overhead per query |
| `ZORDER BY` | Co-locates related rows within compacted files based on filter columns — enables data skipping |

### What gets optimized
| Table | ZORDER columns | Rationale |
|---|---|---|
| `silver.orders_clean` | `order_date`, `branch` | These are the two most common filter dimensions throughout downstream processing |
| `gold.fact_orders` | `date_key`, `branch_key` | Same filtering pattern, applied to fact table for Power BI query speed |

### Time travel demonstration
We also use Delta's `DESCRIBE HISTORY` command to inspect the version history of `fact_orders`. Every Delta table maintains a complete versioned history, allowing point-in-time queries via `VERSION AS OF`. This is a built-in audit and rollback capability unique to the Lakehouse architecture.

### Why this matters
These optimizations are **not visible at the SQL level** — the same query produces the same results before and after — but they substantially reduce query execution time and compute cost. Production data engineering is as much about *how* the data is stored as *what* it contains.

In [0]:
-- STEP 8.1: Optimize the largest tables for query performance
-- Purpose: Compact Parquet files and co-locate related data using Delta's OPTIMIZE and ZORDER commands.
--          These commands rewrite the underlying storage layout for faster reads.
--
-- ZORDER column choice rationale:
--  - order_date / date_key: time-range filtering is the most common pattern
--  - branch / branch_key: branch slicer is the other primary filter

-- Optimize the Silver clean table (the source of all Gold builds)
OPTIMIZE workspace.silver.orders_clean
ZORDER BY (order_date, branch);

-- Optimize the Gold fact table (the table Power BI will query most)
OPTIMIZE workspace.gold.fact_orders
ZORDER BY (date_key, branch_key);

path,metrics
,"List(4, 3, List(40665212, 60091890, 4.9932555E7, 4, 199730220), List(29505741, 77491137, 5.9853775333333336E7, 3, 179561326), 0, List(minCubeSize(107374182400), List(0, 0), List(3, 179561326), 0, List(3, 179561326), 1, null), null, 0, 1, 3, 0, false, 0, 0, 1778078360185, 1778078371268, 8, 1, null, List(0, 0), null, 22, 22, 16715, 0, null, null)"


In [0]:
-- STEP 8.2: Inspect Delta version history (Time Travel demonstration)
-- Purpose: Demonstrate Delta Lake's built-in version control. 
--          Every operation on a Delta table (CREATE, INSERT, UPDATE, OPTIMIZE) is logged as a versioned commit.
--          This enables:
--             - Audit: see exactly what happened and when
--             - Rollback: revert to any prior version
--             - Point-in-time queries: SELECT ... VERSION AS OF N
--
-- This capability is unique to the Lakehouse and not available in either a traditional Data Warehouse or a raw Data Lake.

-- Show the full version history of the fact table
DESCRIBE HISTORY workspace.gold.fact_orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-05-06T14:39:31.000Z,70487227185763,mamdouhfaheem.mf@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [""date_key"",""branch_key""], batchId -> 0)",null,List(2815300312991744),521b90eb-6de3-4d6a-96fa-bf0afdf000b6,0506-140652-8rk28wzj-v2n,0,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 179561326, p25FileSize -> 49321058, numDeletionVectorsRemoved -> 0, minFileSize -> 40665212, numAddedFiles -> 4, maxFileSize -> 60091890, p75FileSize -> 60091890, p50FileSize -> 49652060, numAddedBytes -> 199730220)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-05-06T13:14:43.000Z,70487227185763,mamdouhfaheem.mf@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> Fact table at order grain. Foreign-keyed to dim_date, dim_branch, dim_item, dim_payment. Joined via INT surrogate keys for fast analytical queries., isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2815300312991744),2f2ace73-be58-40e2-bb0d-7ccf725f38de,0506-110929-f9r2btsx-v2n,null,WriteSerializable,false,"Map(numFiles -> 3, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 11058558, numOutputBytes -> 179561326)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


# Step 9: Self-Documenting Tables
Every table in the Gold layer is enriched with column-level comments and metadata properties. When viewed in the Databricks Catalog Explorer, each table reads like a self-explanatory data dictionary.

### What gets documented
| Object | Documentation type |
|---|---|
| Each Gold dimension and fact table | Column-level `COMMENT` on key columns |
| Each Gold table | `TBLPROPERTIES` with project metadata (purpose, refresh frequency, owner) |

### Why bother with comments?
In production data platforms, undocumented tables are a major source of friction:
- New analysts don't know what columns mean
- Decisions get made based on misinterpreted columns
- "Tribal knowledge" creates dependencies on specific people

Self-documenting tables solve all three problems with zero ongoing cost. Anyone who opens the catalog gets the context immediately.

In [0]:
-- STEP 9.2: Add column comments and table properties to Gold tables
-- Purpose: Enrich the Gold layer with detailed metadata for self-documentation in the Databricks Catalog Explorer.   

-- dim_date: column comments
ALTER TABLE workspace.gold.dim_date 
ALTER COLUMN date_key      COMMENT 'Surrogate INT key. Foreign key from fact_orders.date_key.';
ALTER TABLE workspace.gold.dim_date 
ALTER COLUMN full_date     COMMENT 'The actual calendar date. Natural key.';
ALTER TABLE workspace.gold.dim_date 
ALTER COLUMN year          COMMENT 'Calendar year (2020-2025).';
ALTER TABLE workspace.gold.dim_date 
ALTER COLUMN month         COMMENT 'Calendar month (1-12).';
ALTER TABLE workspace.gold.dim_date 
ALTER COLUMN day_name      COMMENT 'Full day name in English (Monday-Sunday).';
ALTER TABLE workspace.gold.dim_date 
ALTER COLUMN is_weekend    COMMENT 'Western convention (1 = Saturday or Sunday). For Egyptian week, use is_weekend_local in kpi_hourly_heatmap.';

-- dim_branch: column comments
ALTER TABLE workspace.gold.dim_branch 
ALTER COLUMN branch_key      COMMENT 'Surrogate INT key. Foreign key from fact_orders.branch_key.';
ALTER TABLE workspace.gold.dim_branch 
ALTER COLUMN branch_name_ar  COMMENT 'Branch name in Arabic (original from source data).';
ALTER TABLE workspace.gold.dim_branch 
ALTER COLUMN branch_name_en  COMMENT 'Branch name in English.';

-- dim_item: column comments
ALTER TABLE workspace.gold.dim_item 
ALTER COLUMN item_key   COMMENT 'Surrogate INT key. Foreign key from fact_orders.item_key.';
ALTER TABLE workspace.gold.dim_item 
ALTER COLUMN category   COMMENT 'Item category in Arabic (e.g., مشويات / Grills, طواجن / Tagines).';
ALTER TABLE workspace.gold.dim_item 
ALTER COLUMN item_name  COMMENT 'Item name in Arabic.';

-- dim_payment: column comments
ALTER TABLE workspace.gold.dim_payment 
ALTER COLUMN payment_key     COMMENT 'Surrogate INT key. Foreign key from fact_orders.payment_key.';
ALTER TABLE workspace.gold.dim_payment 
ALTER COLUMN payment_method  COMMENT 'Payment method: Cash, Card, or Wallet.';

-- fact_orders: column comments
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN unique_order_id           COMMENT 'Globally-unique business key. CONCAT(_source_file, "_", source_order_id).';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN source_order_id           COMMENT 'Original order_id from source file. Not unique across files.';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN customer_id               COMMENT 'Customer identifier (natural key, no dim_customer in this scope).';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN date_key                  COMMENT 'Foreign key to dim_date.';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN branch_key                COMMENT 'Foreign key to dim_branch.';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN item_key                  COMMENT 'Foreign key to dim_item.';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN payment_key               COMMENT 'Foreign key to dim_payment.';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN total_amount              COMMENT 'CORRECTED order total = price * quantity * (1 - discount). Recomputed in Silver layer.';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN total_amount_source       COMMENT 'Original total_amount from source data. Preserved for audit. May differ from total_amount.';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN total_amount_was_corrected COMMENT 'TRUE if Silver layer corrected this row. ~29% of rows are TRUE.';
ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN order_type                COMMENT 'Order channel: Dine-in, Delivery, or Takeaway.';

-- Table properties: project-level metadata for all Gold tables
ALTER TABLE workspace.gold.dim_date    SET TBLPROPERTIES (
    'project'         = 'Restaurant Big Data Analytics',
    'layer'           = 'gold',
    'pattern'         = 'star schema dimension',
    'refresh_pattern' = 'full overwrite from silver.orders_clean'
);

ALTER TABLE workspace.gold.dim_branch  SET TBLPROPERTIES (
    'project'         = 'Restaurant Big Data Analytics',
    'layer'           = 'gold',
    'pattern'         = 'star schema dimension',
    'refresh_pattern' = 'full overwrite from silver.orders_clean'
);

ALTER TABLE workspace.gold.dim_item    SET TBLPROPERTIES (
    'project'         = 'Restaurant Big Data Analytics',
    'layer'           = 'gold',
    'pattern'         = 'star schema dimension',
    'refresh_pattern' = 'full overwrite from silver.orders_clean'
);

ALTER TABLE workspace.gold.dim_payment SET TBLPROPERTIES (
    'project'         = 'Restaurant Big Data Analytics',
    'layer'           = 'gold',
    'pattern'         = 'star schema dimension',
    'refresh_pattern' = 'full overwrite from silver.orders_clean'
);

ALTER TABLE workspace.gold.fact_orders SET TBLPROPERTIES (
    'project'         = 'Restaurant Big Data Analytics',
    'layer'           = 'gold',
    'pattern'         = 'star schema fact',
    'grain'           = 'one row per order',
    'refresh_pattern' = 'full overwrite from silver.orders_clean'
);

-- Verify: see all the comments on fact_orders columns
DESCRIBE TABLE EXTENDED workspace.gold.fact_orders;

col_name,data_type,comment
unique_order_id,string,"Globally-unique business key. CONCAT(_source_file, ""_"", source_order_id)."
source_order_id,bigint,Original order_id from source file. Not unique across files.
customer_id,bigint,"Customer identifier (natural key, no dim_customer in this scope)."
date_key,int,Foreign key to dim_date.
branch_key,int,Foreign key to dim_branch.
item_key,int,Foreign key to dim_item.
payment_key,int,Foreign key to dim_payment.
order_type,string,"Order channel: Dine-in, Delivery, or Takeaway."
hour,int,null
is_weekend,int,null


In [0]:
-- STEP 9.3: Add comments to remaining fact_orders columns
-- Purpose: Fill in column comments missed in Step 9.2 for completeness.

ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN hour            COMMENT 'Hour of day when order was placed (10-23; restaurant operates 14h/day).';

ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN is_weekend      COMMENT 'Western weekend flag (1 = Saturday/Sunday). Use day_name for Egyptian/Saudi week analysis.';

ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN price           COMMENT 'Unit price of the item in EGP.';

ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN quantity        COMMENT 'Number of units ordered.';

ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN discount        COMMENT 'Discount applied as a decimal (0.10 = 10% off). Coalesced from null to 0 in Silver.';

ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN rating          COMMENT 'Customer rating on 1-5 scale.';

ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN _source_file    COMMENT 'Source filename (data lineage). Carried forward from Bronze.';

ALTER TABLE workspace.gold.fact_orders 
ALTER COLUMN _source_format  COMMENT 'Source format: csv or json. Carried forward from Bronze.';

-- Verify: every column should now have a comment
SELECT 
    column_name, 
    data_type, 
    comment 
FROM information_schema.columns 
WHERE table_catalog = 'workspace' 
  AND table_schema  = 'gold' 
  AND table_name    = 'fact_orders'
ORDER BY ordinal_position;

column_name,data_type,comment
unique_order_id,STRING,"Globally-unique business key. CONCAT(_source_file, ""_"", source_order_id)."
source_order_id,LONG,Original order_id from source file. Not unique across files.
customer_id,LONG,"Customer identifier (natural key, no dim_customer in this scope)."
date_key,INT,Foreign key to dim_date.
branch_key,INT,Foreign key to dim_branch.
item_key,INT,Foreign key to dim_item.
payment_key,INT,Foreign key to dim_payment.
order_type,STRING,"Order channel: Dine-in, Delivery, or Takeaway."
hour,INT,Hour of day when order was placed (10-23; restaurant operates 14h/day).
is_weekend,INT,Western weekend flag (1 = Saturday/Sunday). Use day_name for Egyptian/Saudi week analysis.
